# Genome Analysis Pipeline: BLAST → Alignment → Variant Calling
**Organism:** SARS-CoV-2 (Wuhan-Hu-1 reference, NC_045512.2)  
**Pipeline:** BLAST sequence identification → BWA read alignment → SAMtools processing → bcftools variant calling → Python variant analysis

---

## Overview

This notebook implements a complete, end-to-end genome analysis pipeline: starting from unknown sequences with no annotation, identifying them computationally, aligning sequencing reads to a reference genome, calling variants, and interpreting the results in Python.

**The three-stage pipeline:**

```
Unknown sequences
      ↓
  [BLAST] — sequence similarity search against NCBI nr
      ↓
  Identified: SARS-CoV-2 reads + Wuhan reference
      ↓
  [BWA] — align paired-end reads to reference genome
      ↓
  SAM/BAM alignment file
      ↓
  [SAMtools + bcftools] — sort, quality filter, call variants
      ↓
  VCF file of SARS-CoV-2 variants
      ↓
  [Python / pandas / cyvcf2] — parse, filter, and characterize mutations
```

**Why this matters:** this pipeline represents the foundational workflow of resequencing studies — determining how a sequenced sample differs from a known reference genome. For SARS-CoV-2, this approach underlies global genomic surveillance efforts (GISAID, Nextstrain) that tracked the emergence and spread of variants of concern throughout the pandemic.

**Tools used:**

| Tool | Stage | Purpose |
|---|---|---|
| NCBI BLAST+ | Identification | Protein/nucleotide similarity search against NCBI databases |
| BioPython | BLAST parsing | Structured parsing of BLAST XML output |
| BWA | Alignment | Burrows-Wheeler Aligner for short-read alignment to a reference |
| SAMtools | Processing | SAM/BAM conversion, sorting, indexing, and quality filtering |
| bcftools | Variant calling | mpileup-based variant calling and VCF output |
| cyvcf2 + pandas | Analysis | VCF parsing, filtering, and mutation characterization in Python |

## 1. Setup

In [ ]:
# Install command-line alignment and variant calling tools
!apt install -y bwa samtools bcftools ncbi-blast+ -q

# Install Python libraries for parsing BLAST and VCF output
!pip install biopython cyvcf2 -q

print("All tools installed.")
!bwa 2>&1 | head -3
!samtools --version | head -1
!bcftools --version | head -1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Bio.Blast import NCBIXML
import cyvcf2

## 2. Sequence Identification with BLAST

The first step in any resequencing pipeline is confirming the identity of your sequences. Before aligning reads to a reference, we need to know *which* reference to use. BLAST (Basic Local Alignment Search Tool) finds sequences in public databases that are similar to our query, enabling identification even without prior knowledge of the organism.

We start with two unknown protein sequences and run `blastp` against NCBI's non-redundant protein database (`nr`). Output format 6 (`-outfmt 6`) produces a tab-delimited table suitable for downstream analysis.

**BLAST tabular output columns (outfmt 6):**
`query_id`, `subject_id`, `pct_identity`, `alignment_length`, `mismatches`, `gap_openings`, `query_start`, `query_end`, `subject_start`, `subject_end`, `e_value`, `bit_score`

In [ ]:
# Download two unknown sequences for identification
!wget -q 'https://raw.githubusercontent.com/phagenomics/GENE5120/main/Unknown-01.fasta'
!wget -q 'https://raw.githubusercontent.com/phagenomics/GENE5120/main/Unknown-02.fasta'

!echo "Unknown-01.fasta:"
!head -3 Unknown-01.fasta
!echo ""
!echo "Unknown-02.fasta:"
!head -3 Unknown-02.fasta

In [ ]:
# Run blastp against NCBI nr — this queries NCBI's remote servers
# For reproducibility, a pre-downloaded result is available in the class repo
# if NCBI rate-limits requests during this session
print("Running BLAST for Unknown-01 (this may take 1-2 minutes)...")
!blastp -query Unknown-01.fasta -db nr -remote -max_target_seqs 100 \
        -outfmt 6 -out BLASTP_Unknown01.txt

print("Running BLAST for Unknown-02 (four sequences, may take several minutes)...")
!blastp -query Unknown-02.fasta -db nr -remote -max_target_seqs 25 \
        -outfmt 6 -out BLASTP_Unknown02.txt

print("BLAST complete.")

### 2.1 Parse and analyze BLAST results

In [ ]:
BLAST_COLS = ["Query_ID", "Subject_ID", "Pct_Identity", "Aln_Length",
              "Mismatch", "Gap_Openings", "Query_Start", "Query_End",
              "Subject_Start", "Subject_End", "E_Value", "Bit_Score"]

blast1 = pd.read_csv("BLASTP_Unknown01.txt", sep="	", header=None, names=BLAST_COLS)

print(f"Total hits for Unknown-01: {len(blast1)}")
print(f"\nSummary statistics — percent identity:")
print(f"  Mean:    {blast1['Pct_Identity'].mean():.2f}%")
print(f"  Max:     {blast1['Pct_Identity'].max():.2f}%")
print(f"  Min:     {blast1['Pct_Identity'].min():.2f}%")
print(f"\nTop 5 hits (sorted by bit score):")
blast1.sort_values('Bit_Score', ascending=False).head(5)[
    ['Query_ID', 'Subject_ID', 'Pct_Identity', 'Aln_Length', 'E_Value', 'Bit_Score']
]

In [ ]:
blast2 = pd.read_csv("BLASTP_Unknown02.txt", sep="	", header=None, names=BLAST_COLS)

print("Per-query summary for Unknown-02 (four sequences):")
blast2.groupby("Query_ID").agg(
    n_hits=('Subject_ID', 'count'),
    mean_pct_id=('Pct_Identity', 'mean'),
    max_pct_id=('Pct_Identity', 'max'),
    top_bitscore=('Bit_Score', 'max')
).round(2)

In [ ]:
# Top hit per query — the most likely identity of each unknown sequence
top_hits = blast2.sort_values('Bit_Score', ascending=False).groupby('Query_ID').head(1)
print("Top hit per sequence in Unknown-02:")
print(top_hits[['Query_ID', 'Subject_ID', 'Pct_Identity', 'E_Value', 'Bit_Score']].to_string(index=False))

## 3. Download Reference Genome and Build BWA Index

With our sequences identified, we download the corresponding reference genome for alignment. For SARS-CoV-2, this is the original Wuhan-Hu-1 isolate (NCBI accession NC_045512.2), which serves as the canonical reference for all SARS-CoV-2 genomic analysis.

**BWA indexing** builds a Burrows-Wheeler Transform data structure from the reference sequence, enabling rapid alignment of millions of short reads without scanning the full reference for every read.

In [ ]:
# Download SARS-CoV-2 paired-end sequencing reads and Wuhan reference genome
!gdown 1J6nku7fSKE5t9yGgw0m1sUGy6regdmeb  # SARS.R1.fastq
!gdown 1e05YYPAHcKEjNYgwMntknvKY-Je-W2bh  # SARS.R2.fastq
!gdown 1yTOH-9w0RbcBhwetNWzYxFXENtiyPlzW  # WuhanRef.fasta

print("Files downloaded:")
!ls -lh SARS.R1.fastq SARS.R2.fastq WuhanRef.fasta

In [ ]:
# Inspect the reference genome
!head -5 WuhanRef.fasta
!grep -c ">" WuhanRef.fasta   # number of sequences (should be 1 for SARS-CoV-2)

In [ ]:
# Count reads in each FASTQ file (4 lines per read)
r1_reads = int(!wc -l SARS.R1.fastq)
print(f"R1 reads: {r1_reads // 4:,}")
r2_reads = int(!wc -l SARS.R2.fastq)
print(f"R2 reads: {r2_reads // 4:,}")

In [ ]:
# Build the BWA index — creates .bwt, .pac, .ann, .amb, .sa files
print("Building BWA index for WuhanRef.fasta...")
!bwa index -p WuhanRef WuhanRef.fasta
print("Index built:")
!ls -lh WuhanRef.*

## 4. Align Paired-End Reads with BWA

We use **BWA-ALN** followed by **BWA-SAMPE** for paired-end alignment. This two-step approach (`.sai` intermediate files → SAM output) is appropriate for shorter reads (< 70 bp) and smaller genomes like SARS-CoV-2 (30 kb).

For larger genomes (human, mouse) or longer reads, `bwa mem` is the preferred single-step alternative.

**Paired-end alignment workflow:**
1. `bwa aln` — finds the best alignment position for each read independently (→ `.sai` files)
2. `bwa sampe` — combines forward and reverse read alignments, resolving insert size and orientation to produce a proper paired-end SAM file

In [ ]:
# Step 1: Find alignment positions for R1 (forward) reads
print("Aligning R1 reads...")
!bwa aln WuhanRef SARS.R1.fastq > SARS.R1.sai

# Step 2: Find alignment positions for R2 (reverse) reads
print("Aligning R2 reads...")
!bwa aln WuhanRef SARS.R2.fastq > SARS.R2.sai

# Step 3: Combine into a paired-end SAM file
print("Generating paired-end SAM...")
!bwa sampe WuhanRef SARS.R1.sai SARS.R2.sai SARS.R1.fastq SARS.R2.fastq > SARS.sam

print("Alignment complete.")
!ls -lh SARS.sam

In [ ]:
# How many reads mapped to the SARS-CoV-2 genome?
!samtools flagstat SARS.sam

In [ ]:
# Inspect the SAM alignment file format
# Each line: read_name, FLAG, chromosome, position, MAPQ, CIGAR, mate_chr, mate_pos, insert_size, seq, qual, tags
!samtools view SARS.sam | head -5 | cut -f1-11

In [ ]:
# Count uniquely mapped reads to the SARS-CoV-2 chromosome (NC_045512.2)
mapped = !grep 'NC_045512.2' SARS.sam | awk '{print $1}' | sort | uniq | wc -l
print(f"Read pairs mapping to NC_045512.2: {mapped[0]}")

## 5. Convert, Quality Filter, Sort, and Index

Before variant calling, the alignment must be:
1. **Quality filtered** — remove reads with mapping quality (MAPQ) below a threshold. MAPQ is a Phred-scaled probability that the read is mapped to the wrong location; MAPQ ≥ 12 means less than ~6% chance of mismapping
2. **Converted to BAM** — binary compression for efficient storage and random access
3. **Sorted by coordinate** — required by all downstream variant callers
4. **Indexed** — `.bai` index enables tools to jump to any genomic position without scanning the full file

In [ ]:
# Filter by mapping quality, convert to BAM, and sort — all in one pipe
!samtools view -bS -q 12 SARS.sam | samtools sort -o SARS.sorted.bam

# Index the sorted BAM
!samtools index SARS.sorted.bam

print("BAM processing complete.")
!ls -lh SARS.sorted.bam SARS.sorted.bam.bai

In [ ]:
# Final alignment statistics on the quality-filtered, sorted BAM
!samtools flagstat SARS.sorted.bam

## 6. Variant Calling with bcftools

**Variant calling** identifies positions where the sequenced sample differs from the reference genome. We use a two-step bcftools pipeline:

1. **`bcftools mpileup`** — "piles up" all reads covering each position and computes per-base genotype likelihoods. `--max-depth 2000` prevents the likelihood computation from being overwhelmed at extremely high-coverage positions
2. **`bcftools call`** — uses those likelihoods to call variant sites. `--ploidy 1` is appropriate for SARS-CoV-2, which is a haploid virus

The output is a **VCF file** (Variant Call Format) — the universal standard for representing genomic variants.

In [ ]:
# Run variant calling pipeline
print("Running variant calling...")
!bcftools mpileup -f WuhanRef.fasta --max-depth 2000 SARS.sorted.bam | \
    bcftools call --multiallelic-caller --variants-only --ploidy 1 \
    -mv -Oz -o SARS.Variants.vcf.gz

# Decompress for inspection and Python parsing
!gzip -df SARS.Variants.vcf.gz

# Count variants called
n_variants = !grep -v '^#' SARS.Variants.vcf | wc -l
print(f"Total variants called: {n_variants[0]}")

In [ ]:
# Inspect the VCF format
!grep -v '^##' SARS.Variants.vcf | head -10

## 7. Variant Analysis in Python

We parse the VCF into a pandas DataFrame using `cyvcf2` and perform several analyses:
- Filter by quality score to retain high-confidence calls
- Characterize mutation types (transitions vs. transversions)
- Identify high-confidence variants most likely to represent true SARS-CoV-2 mutations

**Transition vs. transversion:**
- **Transitions** — purine↔purine (A↔G) or pyrimidine↔pyrimidine (C↔T); chemically favored, more common as true mutations
- **Transversions** — purine↔pyrimidine (A/G↔C/T); less common, more likely to be sequencing errors at lower quality

In SARS-CoV-2, C→T transitions are particularly notable as they are associated with APOBEC3-driven RNA editing, a host antiviral mechanism.

In [ ]:
# Parse VCF into a DataFrame
rows = []
for variant in cyvcf2.VCF('SARS.Variants.vcf'):
    rows.append({
        'CHROM':    variant.CHROM,
        'POS':      variant.POS,
        'REF':      variant.REF,
        'ALT':      ','.join(variant.ALT),
        'QUAL':     variant.QUAL,
        'GENOTYPE': variant.gt_types[0]
    })

df = pd.DataFrame(rows)

# Map numeric genotype codes to readable labels
genotype_map = {0: 'HOM_REF', 1: 'HET', 2: 'UNKNOWN', 3: 'HOM_ALT'}
df['GENOTYPE_LABEL'] = df['GENOTYPE'].map(genotype_map)

print(f"Total variants: {len(df)}")
print(f"\nGenotype breakdown:")
print(df['GENOTYPE_LABEL'].value_counts())
df.head()

In [ ]:
# Filter to high-confidence variants (QUAL > 200)
df_hq = df[df['QUAL'] > 200].copy()
print(f"High-confidence variants (QUAL > 200): {len(df_hq)}")
df_hq

In [ ]:
# Classify mutation types
TRANSITIONS = {('A','G'), ('G','A'), ('C','T'), ('T','C')}

def mutation_type(ref, alt):
    if len(ref) != 1 or len(alt) != 1:
        return 'INDEL'
    pair = (ref, alt)
    return 'Transition' if pair in TRANSITIONS else 'Transversion'

df_hq['Mutation_Type'] = df_hq.apply(
    lambda r: mutation_type(r['REF'], r['ALT']), axis=1
)
df_hq['Change'] = df_hq['REF'] + '→' + df_hq['ALT']

print("Mutation type breakdown:")
print(df_hq['Mutation_Type'].value_counts())
print("\nSpecific changes:")
print(df_hq['Change'].value_counts())

In [ ]:
# Visualize variant positions and quality scores along the SARS-CoV-2 genome
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Position and quality
axes[0].scatter(df_hq['POS'], df_hq['QUAL'],
                c=['steelblue' if t == 'Transition' else 'darkorange'
                   for t in df_hq['Mutation_Type']],
                s=80, alpha=0.8)
axes[0].set_ylabel('Quality Score (QUAL)')
axes[0].set_title('High-Confidence SARS-CoV-2 Variants vs. Wuhan Reference')
axes[0].axhline(200, color='red', linestyle='--', alpha=0.5, label='QUAL=200 threshold')
axes[0].legend(['Transition', 'Transversion', 'Threshold'])

# Mutation type by position
colors = {'Transition': 'steelblue', 'Transversion': 'darkorange', 'INDEL': 'seagreen'}
for mtype, group in df_hq.groupby('Mutation_Type'):
    axes[1].scatter(group['POS'], [1] * len(group),
                    c=colors.get(mtype, 'gray'), label=mtype, s=80)
axes[1].set_xlabel('Position on SARS-CoV-2 genome (NC_045512.2)')
axes[1].set_ylabel('Variant')
axes[1].set_yticks([])
axes[1].legend()

# Annotate known SARS-CoV-2 gene boundaries
gene_boundaries = {
    'ORF1ab': (266, 21555), 'S': (21563, 25384),
    'ORF3a': (25393, 26220), 'E': (26245, 26472),
    'M': (26523, 27191), 'N': (28274, 29533)
}
for gene, (start, end) in gene_boundaries.items():
    axes[1].axvspan(start, end, alpha=0.08, label=gene)

plt.tight_layout()
plt.show()

In [ ]:
# Transition/transversion ratio (Ti/Tv) — a quality metric for variant calls
ti = (df_hq['Mutation_Type'] == 'Transition').sum()
tv = (df_hq['Mutation_Type'] == 'Transversion').sum()

print(f"Transitions:   {ti}")
print(f"Transversions: {tv}")
if tv > 0:
    print(f"Ti/Tv ratio:   {ti/tv:.2f}")
    print("\n(Expected Ti/Tv for whole-genome sequencing: ~2.0-2.1)")
    print("(Higher Ti/Tv = higher quality variant calls)")
else:
    print("All high-confidence variants are transitions")

print(f"\nC→T transitions (potential APOBEC3 signature): {(df_hq['Change'] == 'C→T').sum()}")

## 8. Summary

This notebook implemented a complete genome resequencing pipeline from unknown sequences through to characterized variants:

**Stage 1 — BLAST identification:**
- Queried unknown protein sequences against NCBI nr using command-line BLAST
- Parsed tabular BLAST output into pandas for statistical summary
- Identified top hits by bit score and computed per-query identity statistics

**Stage 2 — BWA alignment:**
- Downloaded SARS-CoV-2 paired-end reads and the Wuhan reference genome
- Built a BWA index from the reference
- Aligned paired-end reads using `bwa aln` + `bwa sampe`, appropriate for short reads against a small viral genome

**Stage 3 — Variant calling and analysis:**
- Quality filtered (MAPQ ≥ 12), converted to BAM, sorted, and indexed using SAMtools
- Called variants using `bcftools mpileup | bcftools call` with ploidy=1 (haploid virus)
- Parsed VCF into pandas via cyvcf2, applied quality filtering (QUAL > 200)
- Classified mutations as transitions vs. transversions, computed Ti/Tv ratio
- Visualized variant positions along the SARS-CoV-2 genome, annotated with gene boundaries

**Key takeaways:**
- BLAST provides rapid identification of unknown sequences before committing to a reference genome for alignment — skipping this step risks aligning to the wrong reference
- BWA-ALN vs. BWA-MEM: use `bwa aln`/`bwa sampe` for short reads (< 70 bp) and small genomes; use `bwa mem` for longer reads or larger genomes
- Mapping quality (MAPQ) filtering is essential before variant calling — low MAPQ reads aligned to repetitive regions produce spurious variants
- Transition/transversion ratio is a useful quality metric: high Ti/Tv (> 2) indicates the variant calls are dominated by true biological mutations rather than sequencing errors
- C→T transitions at high frequency in a viral genome can indicate host antiviral activity (APOBEC3 editing), adding biological interpretability beyond simply listing variants

**Potential extensions:**
- Annotate variants with their genomic context (which gene, synonymous vs. nonsynonymous) using a GFF annotation file
- Compare variant profiles across multiple SARS-CoV-2 samples to identify lineage-defining mutations
- Apply the same pipeline to a human genome sample, switching to `bwa mem` and adjusting ploidy to 2